#### Error Mitigated QSVM (ZNE) - Spambase - Consistent with Ideal/Noisy

In [ ]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

In [ ]:
# To ensure reproducibility of results
from qiskit_machine_learning.utils import algorithm_globals
algorithm_globals.random_seed = 12345

In [ ]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, recall_score, balanced_accuracy_score

In [ ]:
# --- Qiskit Imports ---
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

##### Load Dataset

In [ ]:
# --- Import Spambase Column Names ---
spambase_columns = [
    "word_freq_make", "word_freq_address", "word_freq_all", "word_freq_3d", "word_freq_our",
    "word_freq_over", "word_freq_remove", "word_freq_internet", "word_freq_order", "word_freq_mail",
    "word_freq_receive", "word_freq_will", "word_freq_people", "word_freq_report", "word_freq_addresses",
    "word_freq_free", "word_freq_business", "word_freq_email", "word_freq_you", "word_freq_credit",
    "word_freq_your", "word_freq_font", "word_freq_000", "word_freq_money", "word_freq_hp",
    "word_freq_hpl", "word_freq_george", "word_freq_650", "word_freq_lab", "word_freq_labs",
    "word_freq_telnet", "word_freq_857", "word_freq_data", "word_freq_415", "word_freq_85",
    "word_freq_technology", "word_freq_1999", "word_freq_parts", "word_freq_pm", "word_freq_direct",
    "word_freq_cs", "word_freq_meeting", "word_freq_original", "word_freq_project", "word_freq_re",
    "word_freq_edu", "word_freq_table", "word_freq_conference", "char_freq_;", "char_freq_(",
    "char_freq_[", "char_freq_!", "char_freq_$", "char_freq_#", "capital_run_length_average",
    "capital_run_length_longest", "capital_run_length_total", "label"
]

# --- 1. Load the Spambase Dataset (LOCAL PATH) ---
# file_path = '/kaggle/input/spambase/spambase.data'
file_path = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\spambase\spambase.data'
df = pd.read_csv(file_path, header=None, names=spambase_columns)
df.drop_duplicates(inplace=True)

print(f"Dataset loaded: {df.shape[0]} samples, {df.shape[1]} features")

##### ZNE Noise Model Factory Function

In [ ]:
# Base error rates (realistic NISQ device)
P_GATE_1Q = 0.001   # 0.1% error for single-qubit gates
P_GATE_2Q = 0.01    # 1.0% error for two-qubit gates
P_READOUT = 0.02    # 2.0% readout error

def get_scaled_noise_model(scale_factor=1.0):
    """
    Build a noise model with scaled error probabilities for ZNE.
    Uses formula: p_scaled = 1 - (1-p)^scale_factor
    
    Returns: (noise_model, backend, pass_manager)
    """
    p_1q_scaled = 1 - (1 - P_GATE_1Q)**scale_factor
    p_2q_scaled = 1 - (1 - P_GATE_2Q)**scale_factor
    p_ro_scaled = 1 - (1 - P_READOUT)**scale_factor
    
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_1q_scaled, 1), ['u1', 'u2', 'u3'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_2q_scaled, 2), ['cx'])
    readout_error = ReadoutError([[1 - p_ro_scaled, p_ro_scaled], [p_ro_scaled, 1 - p_ro_scaled]])
    noise_model.add_all_qubit_readout_error(readout_error)
    
    backend = AerSimulator(noise_model=noise_model, seed_simulator=12345)
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    
    return noise_model, backend, pm

print("ZNE Noise model factory function ready!")
print(f"Base noise: 1q={P_GATE_1Q*100:.2f}%, 2q={P_GATE_2Q*100:.2f}%, readout={P_READOUT*100:.2f}%")

##### Experiment Configurations

In [ ]:
# ==========================================
# EXPERIMENT CONFIGURATIONS (Same as Ideal/Noisy)
# ==========================================
# ZNE uses Richardson extrapolation with scale factors [1.0, 3.0]

experiments = [
    # --- EXP 1: Sample Size Effect (Generalization) ---
    {'id': 'Exp1_100samp',  'samples': 100, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp1_200samp',  'samples': 200, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp1_300samp',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp1_300samp',  'samples': 400, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp1_500samp',  'samples': 500, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear'},

    # --- EXP 2: Dimensionality Effect (Quantum Complexity) ---
    {'id': 'Exp2_2feat',   'samples': 300, 'k_features': 2,  'shots': 1024, 'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp2_4feat',   'samples': 300, 'k_features': 4,  'shots': 1024, 'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp2_6feat',  'samples': 300, 'k_features': 6, 'shots': 1024, 'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp2_8feat',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp2_10feat',  'samples': 300, 'k_features': 10, 'shots': 1024, 'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp2_12feat',  'samples': 300, 'k_features': 12, 'shots': 1024, 'reps': 1, 'entanglement': 'linear'},

    # --- EXP 3: Shot Noise Effect (Measurement Precision) ---
    {'id': 'Exp3_128shots',  'samples': 300, 'k_features': 8, 'shots': 128,  'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp3_512shots',  'samples': 300, 'k_features': 8, 'shots': 512,  'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp3_1024shots', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear'},

    # --- EXP 4: Reps Effect (Circuit Complexity) ---
    {'id': 'Exp4_Reps1', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1,  'entanglement': 'linear'},  
    {'id': 'Exp4_Reps2', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 2,  'entanglement': 'linear'},  
    {'id': 'Exp4_Reps3', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 3,  'entanglement': 'linear'},

    # --- EXP 5: Entanglement Ablation ---
    {'id': 'Exp5_Linear',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp5_Circular', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'circular'},
    {'id': 'Exp5_Full',    'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'full'},
]

print(f"Total experiments configured: {len(experiments)}")
print("Each experiment runs ZNE with scale factors [1.0, 3.0] - Richardson extrapolation")
print("Running might take hours or even several days (2x kernel computations per experiment)")


##### Main Experiment Loop (with ZNE)

In [ ]:
for i, config in enumerate(experiments, 1):
    print("="*80)
    print(f"EXPERIMENT {i}/{len(experiments)}: {config['id']} (ZNE Mitigated)")
    print("="*80)
    print(f"  Samples: {config['samples']}")
    print(f"  K Features: {config['k_features']}")
    print(f"  Shots: {config['shots']}")
    print(f"  Reps: {config['reps']}")
    print(f"  Entanglement: {config['entanglement']}")
    print(f"  Error Mitigation: ZNE (Richardson, scales=[1.0, 3.0])")
    print("="*80)
    
    # --- 1. Data Preparation (Same as Ideal/Noisy) ---
    X = df.drop('label', axis=1)
    y = df['label']
    
    # Split train/test (Keep stratified)
    X_subset, _, y_subset, _ = train_test_split(
        X, y,
        train_size=config['samples'],
        stratify=y,
        random_state=42
    )
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        test_size=0.30,
        random_state=42,
        stratify=y_subset
    )
    
    print(f"\nDataset created: {X_train.shape[0]} train, {X_test.shape[0]} test")
    
    # Standard Scaling
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)
    print("Data scaled successfully")
    
    # Drop Highly Correlated Features (Correlation Analysis)
    corr_matrix = X_train_scaled.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
    
    X_train_scaled.drop(columns=to_drop, inplace=True)
    X_test_scaled.drop(columns=to_drop, inplace=True)
    print(f"Dropped {len(to_drop)} highly correlated features")
    
    # SelectKBest (Same as Ideal/Noisy)
    k = config['k_features']
    selector = SelectKBest(score_func=f_classif, k=k)
    X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
    X_test_kbest = selector.transform(X_test_scaled)
    
    cols = X_train_scaled.columns
    selected_indices = selector.get_support(indices=True)
    selected_features = cols[selected_indices]
    print(f"SelectKBest: Selected {k} features:")
    for idx, name in enumerate(selected_features, 1):
        print(f"     {idx}. {name}")
    
    # --- 2. ZNE: Create Backends for Scale 1.0 and 3.0 ---
    _, backend_1, pm_1 = get_scaled_noise_model(scale_factor=1.0)
    _, backend_3, pm_3 = get_scaled_noise_model(scale_factor=3.0)
    
    sampler_1 = AerSampler.from_backend(backend=backend_1, default_shots=config['shots'])
    sampler_3 = AerSampler.from_backend(backend=backend_3, default_shots=config['shots'])
    
    feature_map = ZZFeatureMap(
        feature_dimension=k, 
        reps=config['reps'], 
        entanglement=config['entanglement']
    )
    
    # Create kernels for both scales
    fidelity_1 = ComputeUncompute(sampler=sampler_1, pass_manager=pm_1)
    fidelity_3 = ComputeUncompute(sampler=sampler_3, pass_manager=pm_3)
    
    qkernel_1 = FidelityQuantumKernel(fidelity=fidelity_1, feature_map=feature_map)
    qkernel_3 = FidelityQuantumKernel(fidelity=fidelity_3, feature_map=feature_map)
    
    print(f"Quantum kernels configured (ZZFeatureMap, reps={config['reps']}, entanglement={config['entanglement']})")
    
    # --- 3. Compute Kernel Matrices at Both Scales ---
    print("\nComputing kernel matrices (scale=1.0)...")
    start_k = time.time()
    kernel_train_1 = qkernel_1.evaluate(x_vec=X_train_kbest)
    kernel_test_1 = qkernel_1.evaluate(x_vec=X_test_kbest, y_vec=X_train_kbest)
    duration_1 = time.time() - start_k
    print(f"  Scale 1.0 computation: {duration_1:.2f}s")
    
    print("Computing kernel matrices (scale=3.0)...")
    start_k = time.time()
    kernel_train_3 = qkernel_3.evaluate(x_vec=X_train_kbest)
    kernel_test_3 = qkernel_3.evaluate(x_vec=X_test_kbest, y_vec=X_train_kbest)
    duration_3 = time.time() - start_k
    print(f"  Scale 3.0 computation: {duration_3:.2f}s")
    
    # --- 4. ZNE: Richardson Extrapolation ---
    # K_mitigated = 1.5 * K(scale=1) - 0.5 * K(scale=3)
    kernel_train_zne = 1.5 * kernel_train_1 - 0.5 * kernel_train_3
    kernel_test_zne = 1.5 * kernel_test_1 - 0.5 * kernel_test_3
    print("ZNE Richardson extrapolation applied.")
    
    # --- 5. Train SVC (Precomputed) ---
    print("\nGrid searching for optimal C...")
    param_grid = {'C': [0.1, 1, 10, 100]}
    svc = SVC(kernel='precomputed', class_weight='balanced')
    grid = GridSearchCV(svc, param_grid, cv=3, scoring='accuracy')
    
    start_t = time.time()
    grid.fit(kernel_train_zne, y_train)
    best_model = grid.best_estimator_
    duration_t = time.time() - start_t
    
    # --- 6. Evaluation ---
    print(f"  → Best C: {grid.best_params_['C']}")
    print(f"  → CV Score: {grid.best_score_:.4f}")
    print(f"  → Training time: {duration_t:.2f}s")
    
    y_train_pred = best_model.predict(kernel_train_zne)
    y_test_pred = best_model.predict(kernel_test_zne)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    bal_acc = balanced_accuracy_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred, pos_label=1)
    gen_gap = abs(train_acc - test_acc)
    
    print(f"  → Train Accuracy: {train_acc:.4f}")
    print(f"  → Test Accuracy: {test_acc:.4f}")
    print(f"  → Test Balanced Accuracy: {bal_acc:.4f}")
    print(f"  → Spam Recall: {recall:.4f}")
    print(f"  → Generalization Gap: {gen_gap:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_test_pred))
    
    # --- 7. Visualization (Mitigated Kernel Heatmap) ---
    plt.figure(figsize=(8, 6))
    plt.imshow(kernel_train_zne, cmap='viridis')
    plt.colorbar()
    plt.title(f"Mitigated Kernel Matrix (Train) - {config['id']}")
    plt.show()
    
    # --- 8. Save Matrices ---
    np.save(f'kernel_train_zne_{config["id"]}.npy', kernel_train_zne)
    np.save(f'kernel_test_zne_{config["id"]}.npy', kernel_test_zne)
    print(f"Saved kernel matrices: kernel_train_zne_{config['id']}.npy, kernel_test_zne_{config['id']}.npy")
    print("\n")